# Object-Oriented Market Making

Audience: Python learners preparing for quant interviews. Prerequisites: basic Python and pandas.

Goals: Quote and trade objects, state management, histories.

Outline: setup, short tasks, checks, pitfalls. Run cells from top to bottom. All market data here is synthetic.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from market_simulator import (MarketMaker, ClientModel, MarketEnvironment,
    QuotedTrade, CompletedTrade, generate_market_data, generate_rfqs, run_simulation, make_quote)
from market_simulator.risk import beta, historical_volatility

# Resolve synthetic data from the installed local package, independent of kernel CWD.
import market_simulator
DATA = Path(market_simulator.__file__).resolve().parents[1] / 'data'
prices = pd.read_csv(DATA / 'market_prices.csv', parse_dates=['timestamp'])
requests = pd.read_csv(DATA / 'client_rfqs.csv', parse_dates=['timestamp'])
print(prices.head(2).to_string(index=False))


## Quote objects

Build a quote with `make_quote` for ticker ALFA, quantity 10, mid 100, timestamp 0 and full spread 0.02. Return its bid and offer as a tuple.


In [ ]:
def object_quote() -> tuple[float, float]:
    q = make_quote('ALFA', 10, 100, 0, .02)
    return q.bid_price, q.offer_price

**Reasoning and complexity:** O(1). QuotedTrade is immutable; record a new quote if prices change.

**Common mistake:** Forgetting the stated side, time ordering, or validation contract. An explicit loop is often a useful alternative when debugging.

In [ ]:
assert object_quote() == (99,101)

## Execute a maker buy

Create a MarketMaker with cash 1000 and limit 20. Add a 10-share quote at bid 99, ask 101, then execute a maker buy at the bid. Return (inventory, cash, history_length).


In [ ]:
def buy_example() -> tuple[int, float, int]:
    mm = MarketMaker(initial_cash=1000, inventory_limit=20)
    q = make_quote('ALFA', 10, 100, 0)
    mm.add_quoted_trade(q)
    mm.add_trade(CompletedTrade('ALFA', 10, q.bid_price, 'buy', 100, q.bid_price, q.offer_price, 0))
    return mm.position('ALFA'), mm.cash, len(mm.completed_trades)

**Reasoning and complexity:** O(1) per trade. A maker buy is a client sell; storing both quote and completed trade preserves the audit trail.

**Common mistake:** Forgetting the stated side, time ordering, or validation contract. An explicit loop is often a useful alternative when debugging.

In [ ]:
assert buy_example() == (10, 10, 1)

## Mark inventory

Given a maker that already traded and a current mark dictionary, return its total P&L using mark_to_market.


In [ ]:
def total_pnl(maker: MarketMaker, marks: dict[str, float]) -> float:
    return maker.mark_to_market(marks)['total_pnl']

**Reasoning and complexity:** O(k) for k open tickers. Equity equals cash plus marked inventory; do not count the purchase price twice.

**Common mistake:** Forgetting the stated side, time ordering, or validation contract. An explicit loop is often a useful alternative when debugging.

In [ ]:
m = MarketMaker(initial_cash=1000)
q = make_quote('A', 1, 100, 0)
m.add_trade(CompletedTrade('A', 1, 99, 'buy', 100, 99, 101, 0))
assert total_pnl(m, {'A':100}) == 1

## Extension
Try a different random seed or add a transaction fee. Explain how your answer changes.